<a href="https://colab.research.google.com/github/ujwalbajaj/cd/blob/main/Ass2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [16]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bison is already the newest version (2:3.8.2+dfsg-1build1).
flex is already the newest version (2.6.4-8build2).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


## Lex only

In [20]:
#@title Writing Lex program
%%writefile program.l

%{
#include <stdio.h>
#include <stdlib.h>

int line_no = 1;
%}

%%

"int"       { printf("KEYWORD       : %s\n", yytext); }
"float"     { printf("KEYWORD       : %s\n", yytext); }
"char"      { printf("KEYWORD       : %s\n", yytext); }
"double"    { printf("KEYWORD       : %s\n", yytext); }
"if"        { printf("KEYWORD       : %s\n", yytext); }
"else"      { printf("KEYWORD       : %s\n", yytext); }
"while"     { printf("KEYWORD       : %s\n", yytext); }
"for"       { printf("KEYWORD       : %s\n", yytext); }
"return"    { printf("KEYWORD       : %s\n", yytext); }
"void"      { printf("KEYWORD       : %s\n", yytext); }

"//".*      { }

[0-9]+[a-zA-Z_][a-zA-Z0-9_]* {
    printf("LEXICAL ERROR : Invalid identifier %s at line %d\n", yytext, line_no);
}

[0-9]+\.[0-9]+ {
    printf("FLOAT         : %s\n", yytext);
}

[0-9]+ {
    printf("INTEGER       : %s\n", yytext);
}

[a-zA-Z_][a-zA-Z0-9_]* {
    printf("IDENTIFIER    : %s\n", yytext);
}

"++" { printf("OPERATOR      : %s\n", yytext); }
"--" { printf("OPERATOR      : %s\n", yytext); }
"==" { printf("OPERATOR      : %s\n", yytext); }
"!=" { printf("OPERATOR      : %s\n", yytext); }
"<=" { printf("OPERATOR      : %s\n", yytext); }
">=" { printf("OPERATOR      : %s\n", yytext); }
"&&" { printf("OPERATOR      : %s\n", yytext); }
"||" { printf("OPERATOR      : %s\n", yytext); }

"+" { printf("OPERATOR      : %s\n", yytext); }
"-" { printf("OPERATOR      : %s\n", yytext); }
"*" { printf("OPERATOR      : %s\n", yytext); }
"/" { printf("OPERATOR      : %s\n", yytext); }
"%" { printf("OPERATOR      : %s\n", yytext); }
"=" { printf("OPERATOR      : %s\n", yytext); }
"<" { printf("OPERATOR      : %s\n", yytext); }
">" { printf("OPERATOR      : %s\n", yytext); }
"!" { printf("OPERATOR      : %s\n", yytext); }

"(" { printf("SEPARATOR     : %s\n", yytext); }
")" { printf("SEPARATOR     : %s\n", yytext); }
"{" { printf("SEPARATOR     : %s\n", yytext); }
"}" { printf("SEPARATOR     : %s\n", yytext); }
"[" { printf("SEPARATOR     : %s\n", yytext); }
"]" { printf("SEPARATOR     : %s\n", yytext); }
";" { printf("SEPARATOR     : %s\n", yytext); }
"," { printf("SEPARATOR     : %s\n", yytext); }

[ \t\r]+ { }

\n { line_no++; }

. {
    printf("LEXICAL ERROR : Invalid character %s at line %d\n", yytext, line_no);
}

%%

int yywrap()
{
    return 1;
}

int main(int argc, char *argv[])
{
    if (argc != 2)
    {
        printf("Usage: ./a.out <FILENAME>\n");
        return 1;
    }

    yyin = fopen(argv[1], "r");

    if (yyin == NULL)
    {
        printf("Cannot open file %s\n", argv[1]);
        return 1;
    }

    printf("LEXICAL ANALYZER OUTPUT\n");
    printf("=======================\n");

    yylex();

    fclose(yyin);

    return 0;
}

Overwriting program.l


if you want to use at txt as an input

In [22]:
%%writefile program.txt

int main() {
    int count = 10;
    float avg = 25.50;

    count++;
    avg = avg + 2.5;

    2value = 20;

    @count = count + 1;

    if (count >= 10) {
        return 0;
    }

    return 0;
}

Overwriting program.txt


In [23]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex -l program.l
gcc lex.yy.c
./a.out program.txt

LEXICAL ANALYZER OUTPUT
KEYWORD       : int
IDENTIFIER    : main
SEPARATOR     : (
SEPARATOR     : )
SEPARATOR     : {
KEYWORD       : int
IDENTIFIER    : count
OPERATOR      : =
INTEGER       : 10
SEPARATOR     : ;
KEYWORD       : float
IDENTIFIER    : avg
OPERATOR      : =
FLOAT         : 25.50
SEPARATOR     : ;
IDENTIFIER    : count
OPERATOR      : ++
SEPARATOR     : ;
IDENTIFIER    : avg
OPERATOR      : =
IDENTIFIER    : avg
OPERATOR      : +
FLOAT         : 2.5
SEPARATOR     : ;
LEXICAL ERROR : Invalid identifier 2value at line 9
OPERATOR      : =
INTEGER       : 20
SEPARATOR     : ;
LEXICAL ERROR : Invalid character @ at line 11
IDENTIFIER    : count
OPERATOR      : =
IDENTIFIER    : count
OPERATOR      : +
INTEGER       : 1
SEPARATOR     : ;
KEYWORD       : if
SEPARATOR     : (
IDENTIFIER    : count
OPERATOR      : >=
INTEGER       : 10
SEPARATOR     : )
SEPARATOR     : {
KEYWORD       : return
INTEGER       : 0
SEPARATOR     : ;
SEPARATOR     : }
KEYWORD       : return
INTEGER 

## Lex and Yacc combined

In [ ]:
#@title Writing YACC program
%%writefile program.y

%{
    #include<stdio.h>
    #include<stdlib.h>
%}
%token DIGIT LETTER UND NL
%%
stmt: variable NL {printf("Valid Identifier\n");exit(0);}
variable: LETTER alphanumeric;
alphanumeric: LETTER alphanumeric
            | DIGIT alphanumeric
            | UND alphanumeric
            | LETTER
            | DIGIT
            | UND;
%%

int yyerror(){
    printf("Invalid Identifier\n");
    exit(0);
}

void main(){
    printf("Enter the variable name: ");
    yyparse();
}

Overwriting program.y


In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include "y.tab.h"
%}
%%
[a-zA-Z] {return LETTER;}
[0-9] {return DIGIT;}
[_] {return UND;}
\n {return NL;}
. {return yytext[0];}
%%

Overwriting program.l


if you want to use at txt as an input

In [ ]:
%%writefile program.txt

This is a sample file.

In [ ]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll
./a.out

y.tab.c: In function ‘yyparse’:
y.tab.c:1121:16: warning: implicit declaration of function ‘yylex’ [-Wimplicit-function-declaration]
       yychar = yylex ();
                ^~~~~
y.tab.c:1256:7: warning: implicit declaration of function ‘yyerror’; did you mean ‘yyerrok’? [-Wimplicit-function-declaration]
       yyerror (YY_("syntax error"));
       ^~~~~~~
       yyerrok
Enter the variable name: variable_name
Valid Identifier
